# WORKSHOP 3: GOLD CUT — ANALYTIC TABLES AND KPIS

**Lakehouse** : `Lakehouse_gold` · **Notebook** : `NB_03_Gold`
**Objective**: build **6 analytical tables** ready for Power BI.

| Table | Contenu |
|---|---|
| `daily_consumption_by_site` | consommation quotidienne par site |
| `site_kpis` | KPIs mensuels par site |
| `consumption_by_site_type` | aggregates by site type |
| `curtailment_opportunities` | erasure opportunities and signals |
| `monthly_consumption_pivot` | monthly summary |
| `active_alerts` | prioritised alerts |

## Prerequisite

1. Workshops **Bronze** and **Silver**
2. Lakehouse **`Lakehouse_gold`** created, this notebook attached with:
   - `Lakehouse_gold`in lakehouse **by default** (read/write)
   - `Lakehouse_silver`and`Lakehouse_bronze`in lakehouses **secondary** (reading)

---

## Cell 1: Data Exploration Silver available

- **But** : valider que `consumption_enriched`(Silver) is legible
- **Checks**: the **6 sites** of the park via their`site_id`

In [ ]:
# Lit la table Silver et ne garde que la colonne site_id
# distinct() supprime les doublons : on obtient la liste des sites
df_sites = spark.table("Lakehouse_silver.silver.consumption_enriched").select("site_id").distinct()

print(f"📋 Sites présents dans Silver : {df_sites.count()}")
df_sites.orderBy("site_id").show()

**Extended**

- **6 sites** present in Silver
- **`site_id`** : grouping key of all Gold tables

## Cell 2: Verification of Silver source tables

- **Integrity control** before building Gold
- **`consumption_predictions`**: must contain 1 line per hourly measurement
- **`sites_reference`**: must contain 1 line per site

In [ ]:
# count() = nombre de lignes de chaque table source
t1 = spark.table("Lakehouse_silver.silver.consumption_predictions").count()   # prédictions (Silver)
t2 = spark.table("Lakehouse_bronze.bronze.sites_reference").count()           # référentiel des sites (Bronze)

print(f"✅ consumption_predictions : {t1:,} lignes")
print(f"✅ sites_reference         : {t2} lignes")
print("➡️  Prêt pour l'atelier Gold")

**Extended**

- **4,320 predictions** = 6 sites × 720 h ✓
- **6 sites** in the repository ✓
- **Integrity** OK: secure joints

## Cell 3: Creating the gold schema

- **Schema`gold`**: includes the 6 analytical tables
- **`IF NOT EXISTS`**: **idempotent**, reexecutable without error

In [ ]:
# Crée le schéma "gold" dans le lakehouse par défaut s'il n'existe pas encore
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("✅ Schéma gold prêt")

**Extended**

- **Schema`gold`** created in the default lakehouse (`Lakehouse_gold`)
- **Tables** accessibles sous `gold.<table>`

## Cell 4: Import of PySpark libraries

- **`pyspark.sql.functions as F`**: aggregations, conditions, dates
- Unique ** import**, used by all the following cells

In [ ]:
# Fonctions Spark (sum, avg, when, col, round...) importées sous l'alias F
from pyspark.sql import functions as F
print("✅ Fonctions PySpark importées")

**Extended**

- **`F`** disponible : `sum`, `avg`, `max`, `when`, `round`, `to_date`…
- **No calculation output**: only the confirmation message is displayed

## Cell 5: Loading consumption enriched

- **Charge** Silver table enriched with DataFrame Spark
- **Contenu** : consommation horaire, `baseline_7d_mw`, `anomaly`, `price_eur_mwh`weather
- **`count()`**: validate the load

In [ ]:
# Charge la table Silver enrichie en DataFrame (rien n'est calculé avant le count)
df_enriched = spark.table("Lakehouse_silver.silver.consumption_enriched")
print(f"📊 consumption_enriched : {df_enriched.count():,} lignes | {len(df_enriched.columns)} colonnes")

**Extended**

- **4 320 lines** = 6 sites × 720 h
- **27 columns** : consumption, baseline, anomalies, price, weather
- **Factual Heart** of Gold

## Cell 6: Loading consumption predictions

- ** Charges** predictions and their deviations (`prediction_error_mw`)
- **To**: measure model reliability and filter false positives

In [ ]:
# Charge les prédictions du modèle (une ligne par heure et par site)
df_predictions = spark.table("Lakehouse_silver.silver.consumption_predictions")
print(f"📊 consumption_predictions : {df_predictions.count():,} lignes")

**Extended**

- **4,320 lines**: 100% coverage, no predictions missing
- **To be attached** to`consumption_enriched` (cellule 7)

## Cell 7: Enriched Joint + Predictions (Dataset Gold Source)

- **Beginning**`enriched`+`predictions`on (`hour`,`site_id`)
- **`left join`**: no actual measure lost if prediction is missing
- **Result**: dataset source of all Gold tables

In [ ]:
from pyspark.sql import functions as F

# Jointure sur (hour, site_id) : on ajoute prédiction et erreur à chaque mesure réelle
# left join = on garde toutes les mesures, même sans prédiction
df = df_enriched.join(
    df_predictions.select("hour", "site_id",
                          F.col("prediction").alias("predicted_consumption_mw"),  # renommage pour la clarté
                          "prediction_error_mw"),
    on=["hour", "site_id"],
    how="left"
)
print(f"📊 Dataset Gold source : {df.count():,} lignes (attendu : 4 320)")

# Aperçu des colonnes utiles pour la suite (100 premières lignes)
df.select("hour", "site_id", "avg_consumption_mw",
          "predicted_consumption_mw", "prediction_error_mw", "anomaly", "price_eur_mwh").show(100)

**Extended**

- **4 320 = 4,320**: complete joint, no loss
- **`anomaly`** : `✅ Normal`or`⚠️ ANOMALIE`
- **`prediction_error_mw`**: actual deviation vs predicted
- **`price_eur_mwh`** : hourly spot price (- 32 to 230 €/MWh on the data set)
- **Aggregation loan**

## Cell 8: Daily aggregation by site (df daily + table gold.daily consumption by site)

- **Aggregation** schedule → **day** by site (`df_daily`)
- **Calculations**: sum, mean, peak, prediction gap, price, temperature
- **Beginning**`sites_reference`for`site_type`and`capacity_mw`
- **Usage**: daily operational dashboard

In [ ]:
# --- Étape 1 : ajouter site_type, capacité, région, flexibilité depuis le référentiel ---
df_sites_ref = spark.table("Lakehouse_bronze.bronze.sites_reference")

# Dropper explicitement les colonnes dupliquées de df avant la jointure
cols_to_drop = [c for c in ["site_type", "capacity_mw", "region", "flexible"] if c in df.columns]
df = df.drop(*cols_to_drop)

# Jointure propre
df = df.join(
    df_sites_ref.select("site_id", "site_type", "capacity_mw", "region", "flexible"),
    on="site_id", how="left"
)

# Vérification
print("Colonnes après jointure:", df.columns)

# --- Étape 2 : agréger l'horaire en journalier (une ligne par jour et par site) ---
df_daily = df.groupBy(
    F.to_date("hour").alias("date"),                 # heure -> jour
    "site_id", "site_type", "capacity_mw"
).agg(
    F.sum("avg_consumption_mw").alias("total_consumption_mwh"),   # énergie du jour (somme des 24 h)
    F.avg("avg_consumption_mw").alias("avg_consumption_mw"),      # puissance moyenne
    F.max("max_consumption_mw").alias("peak_consumption_mw"),     # pic de charge
    F.avg("baseline_7d_mw").alias("avg_baseline_mw"),             # référence 7 jours
    F.avg("predicted_consumption_mw").alias("avg_predicted_mw"),  # prédiction moyenne
    F.avg("prediction_error_mw").alias("avg_prediction_error_mw"),
    F.avg("price_eur_mwh").alias("avg_price_eur_mwh"),
    F.max("price_eur_mwh").alias("peak_price_eur_mwh"),
    F.avg("temperature_c").alias("avg_temperature_c"),
    F.sum(F.when(F.col("anomaly") == "⚠️ ANOMALIE", 1).otherwise(0)).alias("nb_anomalies"),  # heures anormales
    F.count("*").alias("nb_measurements")                         # nombre d'heures du jour
).orderBy(F.desc("date"), "site_id")

print(f"✅ df_daily prêt : {df_daily.count()} lignes (attendu : 6 sites × 30 jours = 180)")

**Extended**

- **180 lines** = 6 sites × 30 days
- ** Keys**:`total_consumption_mwh` (somme), `peak_consumption_mw` (max), `avg_price_eur_mwh`
- **Dimensions** : `site_type`, `capacity_mw`for pivots

## Cellule 9 : Sauvegarde de daily_consumption_by_site

- **Delta script**:`gold.daily_consumption_by_site`
- **Delta** : transactions **ACID**, versioning, *time travel*
- **`overwrite`**: re-execution**

In [ ]:
# Écrit df_daily en table Delta ; overwrite = remplace la table à chaque exécution
df_daily.write.mode("overwrite").format("delta").saveAsTable("gold.daily_consumption_by_site")
print(f"✅ gold.daily_consumption_by_site : {df_daily.count()} lignes")

# Aperçu : 6 lignes du jour le plus récent
df_daily.show(6)

**Extended**

- **180 lines** written in Delta
- **Scale**: e.g. 30/01, **SITE IND 001** 
- **Order of magnitude** consistent with capacity (5 MW vs. 0.8 MW)
- **Use**: time series by site in Power BI

## Cellule 10 : KPIs mensuels par site (df_kpis)

- **Monthly summary** per site (`df_kpis`) since dayly
- **KPIs**: total energy, **`load_factor_pct`**, `anomaly_rate_pct`, `prediction_error_pct`, prix moyen
- **Load factor** : < 30 % sous-utilisé · 50-70 % optimal · > 80 % risque de saturation

In [ ]:
# KPIs mensuels : on regroupe les lignes journalières par site
df_kpis = df_daily.groupBy("site_id", "site_type", "capacity_mw").agg(
    F.sum("total_consumption_mwh").alias("total_consumption_month_mwh"),   # énergie du mois
    F.avg("avg_consumption_mw").alias("avg_consumption_mw"),
    # Facteur de charge = puissance moyenne / capacité × 100
    F.round(F.avg("avg_consumption_mw") / F.col("capacity_mw") * 100, 2).alias("load_factor_pct"),
    # Taux d'anomalies = heures anormales / heures mesurées × 100
    F.round(F.sum("nb_anomalies") * 100.0 / F.sum("nb_measurements"), 2).alias("anomaly_rate_pct"),
    F.round(F.avg("avg_prediction_error_mw"), 3).alias("avg_prediction_error_mw"),
    # Erreur de prédiction relative à la consommation moyenne (%)
    F.round(F.avg("avg_prediction_error_mw") / F.avg("avg_consumption_mw") * 100, 1).alias("prediction_error_pct"),
    F.round(F.avg("avg_price_eur_mwh"), 2).alias("avg_electricity_price_eur_mwh"),
    F.countDistinct("date").alias("nb_days")                               # jours couverts
).orderBy(F.desc("total_consumption_month_mwh"))

# Cette cellule ne calcule que le DataFrame : on affiche donc un résumé lisible
print(f"✅ df_kpis calculé : {df_kpis.count()} lignes (1 par site)")
df_kpis.select("site_id", "total_consumption_month_mwh", "load_factor_pct",
               "anomaly_rate_pct", "prediction_error_pct").show(truncate=False)

**Extended**

- **1 line per site** (6 lines)
- **Load factor**: calculated on`capacity_mw`
- **`anomaly_rate_pct`** and **`prediction_error_pct`** : *health check* mensuel

## Cell 11: Enrichment with region and flexibility

- **Enriches** KPIs with`region`and`flexible`(since`sites_reference`)
- **`flexible`**: Can the site participate in** removal** (cells 14-16)?

In [ ]:
# Ajoute la région et le flag "flexible" aux KPIs (jointure sur site_id)
df_sites_ref = spark.table("Lakehouse_bronze.bronze.sites_reference")
df_kpis = df_kpis.join(
    df_sites_ref.select("site_id", "region", "flexible"),
    on="site_id", how="left"
)

# Contrôle : colonnes ajoutées et sites éligibles à l'effacement
print("✅ Colonnes ajoutées : region, flexible")
print("🔌 Sites flexibles :", [r["site_id"] for r in df_kpis.filter("flexible = true").select("site_id").orderBy("site_id").collect()])
df_kpis.select("site_id", "region", "flexible").orderBy("site_id").show(truncate=False)

**Extended**

- **Each site** now carries its **region** and flag **`flexible`**
- **Possible** questions: regional erasure potential, sites contributing to peaks

## Cellule 12 : Sauvegarde gold.site_kpis

- **Writing** of`gold.site_kpis`: 1 line per site
- **Most read** by decision makers (operation, analysts, compliance)

In [ ]:
# Écrit les KPIs en table Delta (overwrite = ré-exécutable)
df_kpis.write.mode("overwrite").format("delta").saveAsTable("gold.site_kpis")
print(f"✅ gold.site_kpis : {df_kpis.count()} lignes")

# Affiche la table complète (6 lignes)
df_kpis.show()

**Extended**

- **Load factor**: **28 to 51%** (industrial, 50%, residential, 28%)
- **Anomalies**: **0-2.2%** (most affected residential)
- **Prediction error**: **28 to 47%** of average consumption → model **not very accurate**
- **`flexible`**: IND and RES =`true`, COM = `false`
- **Alimente**: monthly dashboard, compliance, intersite benchmark

## Cellule 13 : Consommation par type de site (gold.consumption_by_site_type)

- **Granularity change**: by **type of site** (Industry / Commercial / Residential) and by day
- **Calculations**: consumption, total capacity, **load factor** segment, average price, number of sites
- **Use**: compare the 3 segments

In [ ]:
# Agrégation par jour et par type de site (Industrie / Commercial / Résidentiel)
df_by_type = df.groupBy(
    F.to_date("hour").alias("date"),
    "site_type"
).agg(
    F.sum("avg_consumption_mw").alias("total_consumption_mwh"),
    F.avg("avg_consumption_mw").alias("avg_consumption_mw"),
    F.sum("capacity_mw").alias("total_capacity_mw"),    # capacité cumulée (ligne horaire × sites)
    # Load factor du segment = consommation / capacité × 100
    F.round(F.sum("avg_consumption_mw") / F.sum("capacity_mw") * 100, 2).alias("load_factor_pct"),
    F.avg("price_eur_mwh").alias("avg_price_eur_mwh"),
    F.countDistinct("site_id").alias("nb_sites")        # sites dans le segment
).orderBy(F.desc("date"), "site_type")

# Sauvegarde Delta
df_by_type.write.mode("overwrite").format("delta").saveAsTable("gold.consumption_by_site_type")
print(f"✅ gold.consumption_by_site_type : {df_by_type.count()} lignes (attendu : 3 types × 30 jours = 90)")

# Aperçu des 3 segments pour le dernier jour
df_by_type.show(3)

**Extended**

- **90 lines** = 3 types × 30 days
- **Colonnes** : consommation, `total_capacity_mw`, `load_factor_pct`, prix moyen, `nb_sites` (= 2)
- **Contrast**: e.g. 30/01: Industry, 58% load factor vs. Residential, 28%
- **Usage**: 3 superimposed curves in Power BI

## Cell 14: Filtering flexible sites eligible for erasure

- **Efficiency (currency)**: voluntary and temporary reduction in consumption
- **Filtre 1** : `flexible == True`
- **Filtre 2** : `avg_consumption_mw > 0.5`(noise avoidance)
- **Sortie** : `df_flex`, cell inlet 15-16

In [ ]:
# Ne garde que les sites flexibles ET une consommation > 0,5 MW (évite le bruit)
df_flex = df.filter(
    (F.col("flexible") == True) &
    (F.col("avg_consumption_mw") > 0.5)
)
print(f"📊 Lignes éligibles effacement : {df_flex.count():,}")

# Contrôle : quels sites sont retenus et combien de lignes chacun
df_flex.groupBy("site_id").count().orderBy("site_id").show()

**Extended**

- **1 479 lines** eligible
- **4 sites** `flexible`(IND + RES) and consumption > 0.5 MW
- **COM** exclus (non flexibles)

## Cell 15: Calculation of erasure potential (curtailing)

- **`ratio_vs_baseline`** : consommation / baseline 7 j (> 1 = surcharge)
- **`curtailment_potential_mw`**: **30%** of consumption (operational limit)
- **`potential_gain_eur`**: MW deleted × (spot price - contract price), otherwise 0
- **`action_signal`** : 🔴 > 300 · 🟠 > 200 · 🟢 < 50 · ⚪ sinon (€/MWh)

In [ ]:
# Calcul des indicateurs d'effacement sur chaque ligne éligible
df_curtailment = df_flex.withColumn(
    "ratio_vs_baseline",                                      # >1 = consomme plus que d'habitude
    F.round(F.col("avg_consumption_mw") / F.col("baseline_7d_mw"), 2)
).withColumn(
    "curtailment_potential_mw",                               # puissance effaçable
    F.round(F.col("avg_consumption_mw") * 0.3, 3)         # 30% effacement
).withColumn(
    "potential_gain_eur",                                     # gain seulement si prix spot > seuil contractuel
    F.when(
        F.col("price_eur_mwh") > F.col("curtailment_price_eur_mwh"),
        F.round((F.col("avg_consumption_mw") * 0.3) *
                (F.col("price_eur_mwh") - F.col("curtailment_price_eur_mwh")), 2)
    ).otherwise(0)
).withColumn(
    "action_signal",                                          # recommandation selon le prix spot
    F.when(F.col("price_eur_mwh") > 300, "🔴 EFFACEMENT MAX")
     .when(F.col("price_eur_mwh") > 200, "🟠 EFFACEMENT PARTIEL")
     .when(F.col("price_eur_mwh") < 50,  "🟢 CONSOMMER")
     .otherwise("⚪ Surveiller")
)

# Cette cellule ne fait que calculer : on affiche donc un résumé
print(f"✅ Colonnes ajoutées : ratio_vs_baseline, curtailment_potential_mw, potential_gain_eur, action_signal")
print(f"📊 Heures évaluées : {df_curtailment.count():,}")
print("📊 Répartition des signaux d'action :")
df_curtailment.groupBy("action_signal").count().orderBy(F.desc("count")).show(truncate=False)
gain = df_curtailment.agg(F.round(F.sum("potential_gain_eur"), 0)).collect()[0][0]
print(f"💶 Gain potentiel total : {gain:,.0f} €")

**Extended**

- **Nouvelles colonnes** : `ratio_vs_baseline`, `curtailment_potential_mw`, `potential_gain_eur`, `action_signal`
- **Exemple** : 3,3 MW × 30 % = **0,99 MW** ; prix 230 € − seuil 102 € → gain ≈ **127 €**
- **Thresholds**: max price observed 

## Cellule 16 : Sauvegarde gold.curtailment_opportunities

- **Writing** of`gold.curtailment_opportunities`
- **Tri** par `potential_gain_eur`Declining: Best Hours First
- **Usage**: operational alerts, flexibility contract optimization

In [ ]:
# Sélectionne et ordonne les colonnes utiles, triées par gain décroissant
df_curtailment = df_curtailment.select(
    "hour", "site_id", "site_type", "curtailment_price_eur_mwh",
    "avg_consumption_mw", "baseline_7d_mw", "ratio_vs_baseline",
    "price_eur_mwh", "curtailment_potential_mw", "potential_gain_eur", "action_signal"
).orderBy(F.desc("potential_gain_eur"))

# Sauvegarde Delta
df_curtailment.write.mode("overwrite").format("delta").saveAsTable("gold.curtailment_opportunities")
print(f"✅ gold.curtailment_opportunities : {df_curtailment.count():,} lignes")

# Résumé : heures rentables (gain > 0) et meilleure opportunité
nb_gain = df_curtailment.filter("potential_gain_eur > 0").count()
best = df_curtailment.first()
print(f"💶 Heures avec gain > 0 : {nb_gain:,} sur {df_curtailment.count():,}")
print(f"🏆 Meilleure opportunité : {best['site_id']} le {best['hour']} → {best['potential_gain_eur']} €")

# Aperçu des 50 meilleures heures
df_curtailment.filter("potential_gain_eur > 0").show(50)

**Extended**

- **1 479 lines** written
- **Distribution** : 
- **Gain**: max. *€127/hour** (IND 001, 24/01 to 13h), total potential *€16,100**
- **496 hours** with gain > 0; prices < contractual threshold (gain = 0)

## Cellule 17 : Pivot mensuel (gold.monthly_consumption_pivot)

- **Monthly vote**: 1 line per site (`year`,`month`)
- **Columns**: total energy, average load factor, average price, anomalies
- **Lighter** than`site_kpis`, suitable for cross-visual** Power BI

In [ ]:
# Synthèse mensuelle : une ligne par site, par année et par mois
df_monthly = df_daily.groupBy(
    "site_id", "site_type",
    F.year("date").alias("year"),
    F.month("date").alias("month")
).agg(
    F.sum("total_consumption_mwh").alias("total_mwh"),                    # énergie du mois
    F.avg(F.col("avg_consumption_mw") / F.col("capacity_mw") * 100).alias("avg_load_factor_pct"),
    F.avg("avg_price_eur_mwh").alias("avg_price_eur_mwh"),
    F.sum("nb_anomalies").alias("total_anomalies")                        # heures anormales du mois
).orderBy("site_id")

# Sauvegarde Delta
df_monthly.write.mode("overwrite").format("delta").saveAsTable("gold.monthly_consumption_pivot")
print(f"✅ gold.monthly_consumption_pivot : {df_monthly.count()} lignes")
df_monthly.show()

**Extended**

- **6 lines** : 1 per site, January 2025
- **Energy**: IND 001, 1 801 MWh, RES 002, 122 MWh
- **Prix moyen** ≈ 95,8 €/MWh
- **Anomalies** : **RES_002 = 16**, RES_001 = 14, IND_002 = 4, COM_002 = 2, COM_001 = 0
- **Multi-month**: to reveal seasonality

## Cell 18: Detection of Alert Events

- **3** alert criteria, combined by **OR**:
  - **Anomaly** detected in Silver (z-score)
  - **Prix spot** > 300 €/MWh
  - **Prediction error** > 1 MW
- **Sortie** : `df_alerts`

In [ ]:
# Garde les heures qui remplissent AU MOINS UN des 3 critères (OU logique "|")
df_alerts = df.filter(
    (F.col("anomaly") == "⚠️ ANOMALIE") |          # 1. anomalie détectée en Silver
    (F.col("price_eur_mwh") > 300)        |          # 2. prix spot extrême
    (F.col("prediction_error_mw") > 1.0)             # 3. modèle de prédiction en défaut
)
print(f"📊 Événements d'alerte : {df_alerts.count():,}")

# Détail par critère (une heure peut compter dans plusieurs)
print(f"   • anomalies            : {df_alerts.filter(F.col('anomaly') == '⚠️ ANOMALIE').count():,}")
print(f"   • prix > 300 €/MWh     : {df_alerts.filter('price_eur_mwh > 300').count():,}")
print(f"   • erreur prédiction > 1 MW : {df_alerts.filter('prediction_error_mw > 1.0').count():,}")

**Extended**

- **260 alerts** out of 4,320 measures (-6%)
- **36** anomalies
- **0** prix > 300 €/MWh (max ≈ 230)
- **231** prediction errors > 1 MW (including 7 also abnormal)
- **Total** : 36 + 231 − 7 = **260**

## Cell 19: Qualification and prioritization of alerts

- **`alert_type`**: oriented towards the right team (technical, trading, data science)
- **`priority`** :
  - **CRITICAL**: anomaly **and** prices > 300
  - ** HIGH**: anomaly **or** price > 300
  - **MOYENNE**: prediction error alone

In [ ]:
# alert_type : le 1er critère vrai l'emporte (ordre des when)
# priority   : CRITIQUE (anomalie + prix) > HAUTE (l'un des deux) > MOYENNE (erreur de prédiction seule)
df_alerts = df_alerts.withColumn(
    "alert_type",
    F.when(F.col("anomaly") == "⚠️ ANOMALIE", "Consommation anormale")
     .when(F.col("price_eur_mwh") > 300,        "Prix spot très élevé")
     .when(F.col("prediction_error_mw") > 1.0,  "Erreur prédiction élevée")
     .otherwise("Autre")
).withColumn(
    "priority",
    F.when((F.col("anomaly") == "⚠️ ANOMALIE") & (F.col("price_eur_mwh") > 300), "CRITIQUE")
     .when((F.col("anomaly") == "⚠️ ANOMALIE") | (F.col("price_eur_mwh") > 300), "HAUTE")
     .when(F.col("prediction_error_mw") > 1.0, "MOYENNE")
     .otherwise("BASSE")
)

# Cette cellule ne fait que calculer : on affiche la répartition type × priorité
print("✅ Colonnes ajoutées : alert_type, priority")
df_alerts.groupBy("alert_type", "priority").count().orderBy("priority", "alert_type").show(truncate=False)

**Extended**

- **Types**: *Anormal consumption* (36) · *High prediction error* (224)
- **Priorities** : **Haut** 36 · **MOYENNE** 224 · **CRITICAL** 0
- **CRITICAL**: requires a price > 300 €/MWh, never reached here

## Cellule 20 : Sauvegarde gold.active_alerts

- **Writing** of`gold.active_alerts`, sorted by priority and by date
- **Use**: notifications (mail, SMS) and real-time dashboard

In [ ]:
# Colonnes finales (alert_time = heure de l'alerte) puis tri :
# priorité (CRITIQUE d'abord) puis alerte la plus récente
df_alerts = df_alerts.select(
    F.col("hour").alias("alert_time"),
    "site_id", "site_type", "alert_type",
    "avg_consumption_mw", "baseline_7d_mw", "predicted_consumption_mw",
    "price_eur_mwh", "priority"
).orderBy(
    F.when(F.col("priority") == "CRITIQUE", 1)
     .when(F.col("priority") == "HAUTE",    2)
     .when(F.col("priority") == "MOYENNE",  3)
     .otherwise(4),
    F.desc("alert_time")
)

# Sauvegarde Delta
df_alerts.write.mode("overwrite").format("delta").saveAsTable("gold.active_alerts")
print(f"✅ gold.active_alerts : {df_alerts.count():,} lignes")

# Aperçu : 5 premières alertes CRITIQUE/HAUTE (s'il y en a)
df_alerts.filter("priority IN ('CRITIQUE', 'HAUTE')").show(5)

**Extended**

- **260 lines** written
- **Top of Table**: Alerts **Top** (anomalies) most recent
- **Ex.**: SITE RES 002, 30/01 to 04 h: 0.50 MW vs baseline 0.16 MW (3× normal)
- **Action**: check the site; MEDIUM alerts fall under the model

## Cell 21: Final Summary of Gold Tables

- **Final control**: 6 Gold tables exist and are readable
- **Prests** for SQL Endpoint and Power BI

In [ ]:
# Liste des 6 tables Gold à contrôler
tables_gold = [
    "gold.daily_consumption_by_site",
    "gold.site_kpis",
    "gold.consumption_by_site_type",
    "gold.curtailment_opportunities",
    "gold.monthly_consumption_pivot",
    "gold.active_alerts"
]

print("=" * 55)
print("📦 TABLES GOLD — RÉCAPITULATIF FINAL")
print("=" * 55)
for table in tables_gold:
    # Relit chaque table depuis le lakehouse pour confirmer qu'elle existe
    count = spark.table(f"Lakehouse_gold.{table}").count()
    print(f"  ✅ {table:<40} {count:>7,} lignes")
print("=" * 55)
print("➡️  Prochaine étape : NB_06 Visualisation matplotlib")

**Extended**

- **6 Gold tables** delivered:

| Table | Lines | Granularity |
|---|---|---|
| **daily_consumption_by_site** | 180 | jour × site |
| **site_kpis** | 6 | site |
| **consumption_by_site_type** | 90 | jour × type |
| **curtailment_opportunities** | 1 479 | heure × site flexible |
| **monthly_consumption_pivot** | 6 | mois × site |
| **active_alerts** | 260 | event |

- **Next step**: visualization (NB 06) and then Power BI